# 19 - Continuous Actions In, Discrete DQN Actions Out

Train a DQN on [Gymnasium Pendulum-v1](https://gymnasium.farama.org/environments/classic_control/pendulum/).
The environment takes a **continuous vector of shape `(1,)`**, containing torque in `[-2, 2]`.
Its observation is `[cos(theta), sin(theta), angular_velocity]`.

One `DiscretizedActionHead` produces **33 Q-values**. Each discrete category selects a complete row of a
fixed `head.action_vectors` table of shape `[33, 1]`. The table includes both limits and zero. The
environment receives `head.discrete_to_continuous(action_id)`; the transformer receives the **actual numeric
torque** in the next history row. The category ID is used only by the objective, never as a
transformer input. DQN chooses a category by epsilon-greedy selection over Q-values; it does not
need a separate policy head or a continuous actor.

```text
history of (previous torque, reward, current observation)
    -> transformer -> 33 Q-values -> one category ID
    -> head.discrete_to_continuous(ID) -> env.step(continuous vector)
    -> append the applied torque, reward, and next observation
```

This is an executable wiring and training example, not a trained swing-up agent or an in-context
adaptation benchmark. It uses a small transformer built from scratch, a local token vocabulary,
and CPU training: no model downloads, Box2D, or MuJoCo are needed. Install the repository's
`examples` extra first. Each cycle collects complete episodes into replay, then trains with DQN and separate delayed action and value heads. Replay is retained across cycles.

In [ ]:
from copy import deepcopy
import json
from pathlib import Path

import gymnasium as gym
from mouse_gym import EnvConfig, make_env
import numpy as np
from tokenizers import Tokenizer as TextTokenizer
from tokenizers.models import WordLevel
import torch
from transformers import PreTrainedTokenizerFast

from mouse_core import AdamW, polyak_update
from mouse_core.data import DataLoader, Datastore, Tokenizer, save_tokenizer, to_device
from mouse_core.models import (
    Model,
    DecodeCache,
    prepare_decode,
    Backbone,
    Embedding,
    MLP,
    DiscretizedActionHead,
    select_discrete_action,
    load_model,
    save_model,
)
from mouse_core.objectives import DqnObjective, boundary_discount

# Artifacts
SAVE_ARTIFACTS = False
OUTPUT_DIR = Path("runs/pendulum-discrete")

# Environment
MAX_ACTIONS = 33
MAX_STEPS_PER_EPISODE = gym.spec("Pendulum-v1").max_episode_steps  # native limit

# Model
HIDDEN_DIM = 32
BACKBONE_LAYERS = 1
BACKBONE_HEADS = 2
NUM_FREQUENCIES = 4
PROJECTION_LAYERS = 2
HEAD_LAYERS = 1
HEAD_SCALE = 0.1  # final prediction linear's initialization

# Collection and rollout
EPISODES_PER_CYCLE = 2
EPSILON = 0.2

# Training
NUM_CYCLES = 2
TRAIN_STEPS = 2

# Optimizer learning rates
LR_EMBEDDING = 3e-4  # token embeddings and numeric Fourier features
LR_BACKBONE = 3e-4  # backbone
LR_PROJECTION = 3e-4  # projection MLP and its input RMSNorm
LR_HEADS = 3e-4  # prediction heads and their input RMSNorms

# Target interpolation rates
POLYAK_TAU_EMBEDDING = 0.01  # delayed embedding
POLYAK_TAU_BACKBONE = 0.01  # delayed backbone
POLYAK_TAU_PROJECTION = 0.01  # delayed projection
POLYAK_TAU_VALUE = 0.01  # delayed value head
POLYAK_TAU_ACTION = 1.0  # independent action-selection head

# Random seeds
SEED = 7

# Runtime
device = torch.device("cpu")
torch.set_num_threads(min(4, torch.get_num_threads()))
torch.manual_seed(SEED)
action_rng = np.random.default_rng(SEED)


## Build Environment

The environment keeps its continuous `Box` action space. The conversion happens in the collector,
so the model's discrete output and the physical action are both visible.
`head.action_vectors[k]` is a complete action vector, even though Pendulum has just one component.
For a multidimensional environment the same lookup would use a `[K, action_dim]` table, still with
one categorical output over its `K` rows. A Cartesian grid would have the product of the per-axis
bin counts; an explicitly chosen finite list of vectors need not contain every combination.

The table and its ordering are part of the policy. Changing either after training changes what
its outputs mean. The head owns this table as a fixed model buffer. It moves with the model and is saved in the
model checkpoint automatically. No separate action-table file is needed.

In [ ]:
env = make_env(EnvConfig(
    id="Pendulum-v1",
    seed=SEED,
    seed_mode="stream",
))
assert env.action_space.shape == (1,)
action_vectors = torch.linspace(
    float(env.action_space.low[0]), float(env.action_space.high[0]), MAX_ACTIONS,
)[:, None]
assert action_vectors.shape == (MAX_ACTIONS, 1)
assert all(env.action_space.contains(vector.numpy()) for vector in action_vectors)
assert action_vectors[MAX_ACTIONS // 2, 0] == 0.0

## Data Pipeline

Each row contains the transition **into** its observation:

| Row field | Transformer input | DQN objective |
| --- | --- | --- |
| `torque` | Applied continuous torque, as a numeric token | No |
| `observation` | Three numeric tokens | No |
| `reward` | Numeric token | Reward |
| `reset_frame` | Numeric flag: 1 on the initial observation, otherwise 0 | No |
| `action_id` | **No** | Copied to objective column `action` |
| `terminated`, `truncated` | No | Boundary discounts |
| `episode_index` | No | Groups the history and loss by episode |

Pendulum's vector has one component, so the collector stores that physical value as scalar
`torque`. It never replaces it with the category ID. A reset frame has dummy torque/ID values;
its flag marks that no action was applied. DQN uses row `t+1`'s ID and reward with row `t`'s
prediction, so the reset row's dummy ID is never a training target.

The small text vocabulary supplies only **field identities**, such as `torque` and `cos`.
Actual floating-point values travel in numeric tensors. Existing Fourier features turn these
values into additions to the field embeddings. The Fourier windows below are wider than the
physical ranges so opposite physical limits do not coincide at the periodic encoding endpoints.

In [ ]:
vocabulary = {
    "[UNK]": 0, "torque": 1, "cos": 2, "sin": 3,
    "velocity": 4, "reward": 5, "reset": 6, "readout": 7,
}
text_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=TextTokenizer(WordLevel(vocab=vocabulary, unk_token="[UNK]")),
    unk_token="[UNK]",
)

tokenizer = Tokenizer(
    tokenizer=text_tokenizer,
    input_fields=[
        {"type": "numeric", "input_field": "torque", "format": "torque",
         "fourier_min": -4.0, "fourier_max": 4.0},
        {"type": "numeric", "input_field": "observation", "input_index": 0,
         "format": "cos", "fourier_min": -2.0, "fourier_max": 2.0},
        {"type": "numeric", "input_field": "observation", "input_index": 1,
         "format": "sin", "fourier_min": -2.0, "fourier_max": 2.0},
        {"type": "numeric", "input_field": "observation", "input_index": 2,
         "format": "velocity", "fourier_min": -16.0, "fourier_max": 16.0},
        {"type": "numeric", "input_field": "reward", "format": "reward",
         "fourier_min": -32.0, "fourier_max": 32.0},
        {"type": "numeric", "input_field": "reset_frame", "format": "reset",
         "fourier_min": -2.0, "fourier_max": 2.0},
        {"type": "token", "format": "readout", "head_output": True},
    ],
    objective_fields=[
        {"input_field": "action_id", "output_field": "action"},
        {"input_field": "reward"},
        {"input_field": "terminated"},
        {"input_field": "truncated"},
    ],
)

## Build Model

The readout is `Embedding → backbone → select head tokens → projection RMSNorm → projection MLP → prediction RMSNorm → prediction head(s)`. The projection is always enabled in this example. Projection and prediction use separate `MLP` instances; this projection preserves the backbone width, and scalar regression sets `out_features=1`. The model indexes the backbone feature tensor to select head tokens before projections and predictions. `Agent.forward()` defines this flow with ordinary PyTorch calls; subclass `Model` to define your own. The projection and its input RMSNorm use the independent `LR_PROJECTION`. Each prediction head also owns an input RMSNorm. Both prenorms are mandatory. Prediction-head `scale` affects only the final linear layer's initial weight and bias; the projection instance omits `scale`. The delayed agent copies this projection and updates it at `POLYAK_TAU_PROJECTION`.

The single head has one Q-value per table row. Its output is a vector of scores over discrete
choices, not a predicted continuous torque. This is discrete DQN with separate backup action and value heads; the
head gives those discrete choices physical meaning with its built-in table and conversion methods.
The table determines the output width, so there is no separate `out_features` to keep in sync.

`discrete_to_continuous(ids)` indexes the table. `continuous_to_discrete(vectors)` chooses the
nearest row by Euclidean distance, with ties going to the lowest ID. The reverse is exact on table
rows; other continuous inputs are quantized. Both methods support batches and return tensors on
the head's device. The reverse conversion is useful for labelling existing continuous-action data,
but any off-grid labels introduce approximation error. Keep the actual applied vector in history.

The reference training kernel supports CPU backward. This short example recomputes the complete
episode prefix for each decision. Larger runs should use the cached rollout pattern from `03`,
closing caches before optimizer updates. The delayed model copies the backbone, projection, and head;
Polyak updates move those copies toward the online model after each optimizer step.
`DelayedAgent` defines the backup network separately. Its value and action heads are frozen copies of the online value head; `polyak(online=model, value_tau=..., action_tau=..., ...)` updates both from that same online head at independent rates. The objective receives their Q scores as `backup_value_predictions` and `backup_action_predictions`.


In [ ]:
class Agent(Model):
    def __init__(self, *, embedding, backbone, projection, value_head):
        super().__init__()
        self.embedding = embedding
        self.backbone = backbone
        self.projection = projection
        self.value_head = value_head

    def get_action(self, *, out, temperature, num_actions=None):
        action_ids = select_discrete_action(
            scores=out["predictions"]["action_value"], valid=out["head_valid_mask"],
            temperature=temperature, num_actions=num_actions,
        )
        return action_ids

    def forward(self, tokens, *, cache=None, use_cache=False):
        if cache is not None and not use_cache:
            raise ValueError("Passing cache= requires use_cache=True.")
        tensors = tokens.to_tensors(next(self.parameters()).device)
        token_ids = tensors["token_ids"]
        numeric_values = tensors["numeric_values"]
        numeric_mask = tensors["numeric_mask"]
        sequence_ids = tensors["sequence_ids"]
        head_token_ids = tensors["head_token_ids"]
        head_step_ids = tensors["head_step_ids"]
        embed = self.embedding(
            token_ids, numeric_values=numeric_values, numeric_mask=numeric_mask,
        )
        head_valid_mask = None
        if use_cache:
            embed, lengths, head_token_ids, head_valid_mask = prepare_decode(
                embeds=embed, tokens=tokens, sequence_ids=sequence_ids,
                head_token_ids=head_token_ids, head_step_ids=head_step_ids,
            )
            if cache is None:
                cache = DecodeCache(sessions=(self.backbone.decode_session(batch_size=tokens.B),))
            if len(cache.sessions) != 1:
                raise ValueError("expected one backbone cache session")
            features = self.backbone(embed, cache=cache.sessions[0], lengths=lengths)
            head_gather_ids = head_token_ids.unsqueeze(-1).expand(
                *head_token_ids.shape, features.shape[-1],
            )
            h = features.gather(1, head_gather_ids)
        else:
            features = self.backbone(embed, sequence_ids=sequence_ids)
            h = features[head_token_ids]
        predictions = self.head(h=h)
        return {
            "predictions": predictions,
            "last_hidden_state": features,
            "head_token_ids": head_token_ids,
            "head_valid_mask": head_valid_mask,
            "cache": cache,
        }

    def head(self, *, h):
        h = self.projection(h.float())
        action_value_predictions = self.value_head(h)
        return {"action_value": action_value_predictions}


class DelayedAgent(Model):
    def __init__(self, *, embedding, backbone, projection, value_head, action_head):
        super().__init__()
        self.embedding = embedding
        self.backbone = backbone
        self.projection = projection
        self.value_head = value_head
        self.action_head = action_head

    forward = Agent.forward

    def head(self, *, h):
        h = self.projection(h.float())
        backup_value_predictions = self.value_head(h)
        backup_action_predictions = self.action_head(h)
        return {
            "backup_value": backup_value_predictions,
            "backup_action": backup_action_predictions,
        }

    def polyak(self, *, online, embedding_tau, backbone_tau, projection_tau, value_tau, action_tau):
        if not (
            0 <= embedding_tau <= 1
            and 0 <= backbone_tau <= 1
            and 0 <= projection_tau <= 1
            and 0 <= value_tau <= 1
            and 0 <= action_tau <= 1
        ):
            raise ValueError("Polyak rates must be in [0, 1].")
        polyak_update(target=self.embedding, source=online.embedding, tau=embedding_tau)
        polyak_update(target=self.backbone, source=online.backbone, tau=backbone_tau)
        polyak_update(target=self.projection, source=online.projection, tau=projection_tau)
        polyak_update(target=self.value_head, source=online.value_head, tau=value_tau)
        polyak_update(target=self.action_head, source=online.value_head, tau=action_tau)


def make_agent():
    embedding = Embedding(hidden_dim=HIDDEN_DIM, vocab_size=len(vocabulary), num_frequencies=NUM_FREQUENCIES)
    backbone = Backbone(
        architecture="qwen3",
        hidden_dim=HIDDEN_DIM,
        num_layers=BACKBONE_LAYERS,
        num_heads=BACKBONE_HEADS,
        train_kernel="reference",
        decode_kernel="flex",
        dtype=torch.float32,
    )

    projection = MLP(
        in_features=backbone.hidden_dim,
        out_features=backbone.hidden_dim,
        hidden_dim=backbone.hidden_dim,
        num_layers=PROJECTION_LAYERS,
    )

    head = DiscretizedActionHead(
        in_features=backbone.hidden_dim,
        action_vectors=action_vectors,
        hidden_dim=backbone.hidden_dim,
        num_layers=HEAD_LAYERS,
        scale=HEAD_SCALE,
        propagate_gradient=1.0,
    )


    agent = Agent(
        embedding=embedding,
        backbone=backbone,
        projection=projection,
        value_head=head,
    ).train().to(device)
    return agent

model = make_agent()

delayed_model = DelayedAgent(
    embedding=deepcopy(model.embedding).requires_grad_(False),
    backbone=deepcopy(model.backbone).requires_grad_(False),
    projection=deepcopy(model.projection).requires_grad_(False),
    value_head=deepcopy(model.value_head).requires_grad_(False),
    action_head=deepcopy(model.value_head).requires_grad_(False),
).eval()

head = model.value_head
print(model)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters; {MAX_ACTIONS} Q-values")
for action_id in (0, MAX_ACTIONS // 2, MAX_ACTIONS - 1):
    vector = head.discrete_to_continuous(action_id)
    recovered_id = int(head.continuous_to_discrete(vector).item())
    assert recovered_id == action_id
    print(f"category {action_id:2d} -> continuous vector {vector.tolist()} -> category {recovered_id}")

query = torch.tensor([0.31])
nearest_id = head.continuous_to_discrete(query)
print(f"Off-grid {query.tolist()} -> category {nearest_id.item()} -> {head.discrete_to_continuous(nearest_id).tolist()}")

## Collect Rollouts

Mouse-gym's first step and the step after a boundary are reset frames: their passed-in action is
ignored. We request those frames explicitly, then select actual actions only after observing a
state. Epsilon exploration samples a **category ID**, so all executed actions belong to the table.
Each action is decoded once; the same torque sent to the environment is stored in history.

At the final observation, stop collecting that episode. Keep the final row for value bootstrapping.
The next episode starts a separate attention/loss group. Replay retains earlier cycles' episodes.

In [ ]:
def make_row(*, output, episode_index, action_id, torque, reset_frame):
    observation, reward, terminated, truncated, _info = output
    action_id = int(action_id)
    torque = float(torque)
    observation = np.asarray(observation, dtype=np.float32).copy()
    reward = float(reward)
    terminated = bool(terminated)
    truncated = bool(truncated)
    reset_frame = float(reset_frame)
    episode_index = int(episode_index)
    return {
        "action_id": action_id,
        "torque": torque,
        "observation": observation,
        "reward": reward,
        "terminated": terminated,
        "truncated": truncated,
        "reset_frame": reset_frame,
        "episode_index": episode_index,
    }


@torch.no_grad()
def collect_episode(*, episode_index, store):
    model.eval()
    reset_output = env.step(np.zeros(env.action_space.shape, dtype=np.float32))
    assert env.step_index == 0
    history = [make_row(
        output=reset_output, episode_index=episode_index,
        action_id=0, torque=0.0, reset_frame=True,
    )]
    store.append(data=history[0])
    episode_return = 0.0
    while True:
        inputs = tokenizer.pack_rows(rows=[history], continuing=None)
        out = model(inputs)
        q_values = out["predictions"]["action_value"][-1]
        if action_rng.random() < EPSILON:
            action_id = int(action_rng.integers(MAX_ACTIONS))
        else:
            action_id = int(q_values.argmax().item())
        continuous_action = head.discrete_to_continuous(action_id).cpu().numpy()
        assert env.action_space.contains(continuous_action)
        output = env.step(continuous_action)
        row = make_row(
            output=output, episode_index=episode_index,
            action_id=action_id, torque=continuous_action[0], reset_frame=False,
        )
        history.append(row)
        store.append(data=row)
        episode_return += row["reward"]
        if row["terminated"] or row["truncated"]:
            return episode_return, history

## Train

`action_id` is mapped to the objective's integer `action` column. All 33 categories represent
valid vectors, so no action mask is needed. The delayed action head chooses the next category
and the delayed value head evaluates it. No continuous-action maximization or new loss is necessary.

Pendulum ends at a **time limit**, not a terminal physical state. Here `gamma_truncated=1` retains
bootstrapping from the final observation; `gamma_terminated=0` would stop it at a true terminal.
`cross_group_backups="bootstrap"` permits that last-state value. Group boundaries still prevent
attention and transition pairs from crossing into the reset observation of the next episode.

In [ ]:
objective = DqnObjective(
    reward=None,
    value=None,
    discount=boundary_discount(
        gamma_step=0.99, gamma_terminated=0.0, gamma_truncated=1.0,
    ),
    temperature=0.0,
    cross_group_backups="bootstrap",
    gate=None,
)
optimizer = AdamW(
    params=model.parameter_groups(
        lr={
            "embedding": LR_EMBEDDING,
            "backbone": LR_BACKBONE,
            "projection": LR_PROJECTION,
            "value_head": LR_HEADS,
        },
    ),
    lr=LR_BACKBONE,
    weight_decay=0.0,
)


def run_train(*, store, cycle, num_steps):
    loader = DataLoader(
        stores=[store],
        samples_budget=(MAX_STEPS_PER_EPISODE + 1) * EPISODES_PER_CYCLE,
        batch_size=1,
        sample_field="episode_index",
        transform=tokenizer,
        num_workers=0,
        seed=SEED + cycle,
    )
    try:
        model.train()
        for _ in range(num_steps):
            inputs, objective_data, group_id = loader.next_batch()
            out = model(inputs)
            with torch.no_grad():
                delayed_out = delayed_model(inputs)
            losses, metrics = objective(
                objective_data=to_device(data=objective_data, device=device),
                group_id=group_id.to(device),
                predictions=out["predictions"]["action_value"],
                backup_value_predictions=delayed_out["predictions"]["backup_value"],
                backup_action_predictions=delayed_out["predictions"]["backup_action"],
            )
            loss = torch.stack(tuple(losses.values())).sum()
            assert torch.isfinite(loss)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
            optimizer.step()
            delayed_model.polyak(
                online=model,
                embedding_tau=POLYAK_TAU_EMBEDDING,
                backbone_tau=POLYAK_TAU_BACKBONE,
                projection_tau=POLYAK_TAU_PROJECTION,
                value_tau=POLYAK_TAU_VALUE,
                action_tau=POLYAK_TAU_ACTION,
            )
        loss_value = float(loss.detach())
        return loss_value, metrics
    finally:
        loader.close()

## Run

Each cycle collects complete episodes, then runs `TRAIN_STEPS` optimizer updates.


In [ ]:
store = Datastore(name="pendulum-replay")
training_log = []
try:
    for cycle in range(NUM_CYCLES):
        returns = []
        for episode in range(EPISODES_PER_CYCLE):
            episode_return, last_history = collect_episode(
                episode_index=cycle * EPISODES_PER_CYCLE + episode, store=store,
            )
            returns.append(episode_return)
        loss, metrics = run_train(store=store, cycle=cycle, num_steps=TRAIN_STEPS)
        training_log.append({"cycle": cycle, "mean_return": float(np.mean(returns)), "loss": loss})
        print(
            f"cycle={cycle} | mean return={np.mean(returns):.1f} | "
            f"loss={loss:.4f} | mean Q={float(metrics['q_values_mean']):.3f} | replay rows={len(store)}"
        )
finally:
    env.close()

print(f"Finished {NUM_CYCLES * EPISODES_PER_CYCLE} episodes and {NUM_CYCLES * TRAIN_STEPS} updates.")

## Inspect the two representations

Changing only `action_id` must not change transformer input tokens or numeric values. Changing
torque must change the numeric input even when the category ID is held fixed. This check also
confirms that the recorded physical action agrees with the lookup table.

The Fourier projection is initialized to zero by mouse-core to preserve pretrained embeddings.
It learns during optimization; the check below verifies that it received a nonzero update.

In [ ]:
example = last_history[1]
assert example["torque"] == float(head.discrete_to_continuous(example["action_id"])[0])
assert head.continuous_to_discrete(torch.tensor([example["torque"]])).item() == example["action_id"]
all_ids = torch.arange(head.out_features, device=device)
assert torch.equal(head.continuous_to_discrete(head.discrete_to_continuous(all_ids)), all_ids)
encoded = tokenizer(example)
id_changed = tokenizer({**example, "action_id": (example["action_id"] + 1) % MAX_ACTIONS})
np.testing.assert_array_equal(encoded.token_ids, id_changed.token_ids)
np.testing.assert_array_equal(encoded.values, id_changed.values)
assert encoded.objective_fields["action"] != id_changed.objective_fields["action"]

low = tokenizer({**example, "torque": -2.0})
high = tokenizer({**example, "torque": 2.0})
np.testing.assert_array_equal(low.token_ids, high.token_ids)
assert low.values[0] == -0.5 and high.values[0] == 0.5
assert model.embedding.fourier.proj.weight.detach().abs().sum().item() > 0.0
print({"category_for_loss": example["action_id"], "torque_for_transformer": example["torque"]})
print("Confirmed: continuous numeric input, discrete objective target.")

## Save Locally

`save_model` includes the head's table as a persistent buffer. `load_model` restores the special
head, Q-values, row ordering, and both conversions. Set `SAVE_ARTIFACTS=True` to save a local bundle
and check the table after reloading. The environment settings remain separate from the model.
The local text tokenizer and the mouse-core packing specification are saved separately because
both use a file named `tokenizer.json`.

Load `text_tokenizer/` with `PreTrainedTokenizerFast.from_pretrained` and pass that tokenizer to
`load_tokenizer(repo_id_or_path=.../packing)`. `model.get_action` continues to return discrete IDs;
pass them to the restored head's `discrete_to_continuous` before stepping a continuous environment.


In [ ]:
if SAVE_ARTIFACTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    save_model(model=model, path=OUTPUT_DIR / "model")
    text_tokenizer.save_pretrained(OUTPUT_DIR / "text_tokenizer")
    save_tokenizer(tokenizer=tokenizer, path=OUTPUT_DIR / "packing")
    (OUTPUT_DIR / "environment.json").write_text(json.dumps({
        "id": "Pendulum-v1",
        "max_episode_steps": MAX_STEPS_PER_EPISODE,
        "action_shape": [head.action_dim],
        "action_dtype": "float32",
        "action_columns": ["torque"],
    }, indent=2) + "\n")
    loaded = load_model(
        model=make_agent(),
        repo_id_or_path=str(OUTPUT_DIR / "model"),
    )
    restored_head = loaded.value_head
    assert isinstance(restored_head, DiscretizedActionHead)
    assert torch.equal(restored_head.action_vectors, head.action_vectors.cpu())
    assert torch.equal(restored_head.continuous_to_discrete(restored_head.action_vectors), all_ids.cpu())
    print(f"Saved and reloaded model with action table, plus tokenizers and environment settings: {OUTPUT_DIR.resolve()}")

## What extra machinery did this require?

1. **A `DiscretizedActionHead`:** one Q-value per fixed table row, exact ID-to-vector lookup,
   and nearest-vector quantization in the other direction. The table is a non-trainable buffer
   included in model copies, device moves, and checkpoints.
2. **Both action representations in collection:** the ID goes to DQN's gather operation; the actual
   continuous components go into the next transformer history row. Keep the table ordering fixed
   during collection, optimization, and inference.
3. **Numeric action input fields:** reuse `Tokenizer(type="numeric")` and the embedding's
   `num_frequencies`, just as for continuous observations. No new embedding API is required.

Replay, the delayed agent, DQN, and Polyak updates are existing components; the loss is
unchanged. Arbitrary offline continuous actions can use `continuous_to_discrete`, but their
quantization error remains an approximation in the training targets. For multidimensional actions,
Euclidean distance uses the table's physical units, so coordinate scaling matters. A Cartesian grid
can become large, and housing the table in the head does not make nearby actions share Q-values.
